# Component 1 — Final Evaluation and Research Analysis

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

try:
    from scipy.stats import wilcoxon
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

CURRENT = Path.cwd().resolve()

if CURRENT.name == "notebooks":
    PROJECT_ROOT = CURRENT.parent
elif (CURRENT / "results").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "results").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise RuntimeError(
        f"Could not locate component_1 from {CURRENT}"
    )

BASELINE_DIR = (
    PROJECT_ROOT
    / "results"
    / "baselines"
)

SENTIMENT_DIR = (
    PROJECT_ROOT
    / "results"
    / "sentiment"
)

TRANSFER_DIR = (
    PROJECT_ROOT
    / "results"
    / "transfer"
)

FINAL_DIR = (
    PROJECT_ROOT
    / "results"
    / "final"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BASELINE_METRICS = (
    BASELINE_DIR
    / "baseline_metrics.csv"
)

BASELINE_PRED = (
    BASELINE_DIR
    / "baseline_test_predictions.csv"
)

SENTIMENT_METRICS = (
    SENTIMENT_DIR
    / "sentiment_calibration_metrics.csv"
)

TRANSFER_METRICS = (
    TRANSFER_DIR
    / "transfer_model_metrics.csv"
)

TRANSFER_GROUP = (
    TRANSFER_DIR
    / "transfer_group_metrics.csv"
)

TRANSFER_PRED = (
    TRANSFER_DIR
    / "transfer_test_predictions.csv"
)

required = [
    BASELINE_METRICS,
    BASELINE_PRED,
    SENTIMENT_METRICS,
    TRANSFER_METRICS,
    TRANSFER_GROUP,
    TRANSFER_PRED,
]

missing = [
    path
    for path in required
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Run notebooks 02-04 first. Missing:\n"
        + "\n".join(
            map(str, missing)
        )
    )


In [ ]:

baseline_metrics = pd.read_csv(
    BASELINE_METRICS
)

baseline_pred = pd.read_csv(
    BASELINE_PRED
)

sentiment_metrics = pd.read_csv(
    SENTIMENT_METRICS
)

transfer_metrics = pd.read_csv(
    TRANSFER_METRICS
)

group_metrics = pd.read_csv(
    TRANSFER_GROUP
)

transfer_pred = pd.read_csv(
    TRANSFER_PRED
)

aspect_model_metrics = pd.concat(
    [
        baseline_metrics[
            baseline_metrics[
                "segment"
            ].isin(
                ["phone", "laptop"]
            )
        ],
        transfer_metrics[
            transfer_metrics[
                "segment"
            ].isin(
                ["phone", "laptop"]
            )
        ],
    ],
    ignore_index=True,
)

selected_cols = [
    "model",
    "segment",
    "precision_micro",
    "recall_micro",
    "micro_f1",
    "macro_f1",
    "exact_match",
    "hamming_loss",
]

final_comparison = (
    aspect_model_metrics[
        selected_cols
    ]
    .copy()
)

display(
    final_comparison.sort_values(
        ["segment", "model"]
    )
)


In [ ]:

gain_rows = []

for domain in [
    "phone",
    "laptop",
]:
    subset = (
        final_comparison[
            final_comparison[
                "segment"
            ].eq(domain)
        ]
        .set_index("model")
    )

    if "b4" not in subset.index:
        continue

    for baseline in [
        "b1",
        "b2",
        "b3",
    ]:
        if baseline not in subset.index:
            continue

        b4_micro = (
            subset.loc[
                "b4",
                "micro_f1",
            ]
        )

        base_micro = (
            subset.loc[
                baseline,
                "micro_f1",
            ]
        )

        b4_macro = (
            subset.loc[
                "b4",
                "macro_f1",
            ]
        )

        base_macro = (
            subset.loc[
                baseline,
                "macro_f1",
            ]
        )

        macro_gain = (
            b4_macro - base_macro
            if pd.notna(b4_macro)
            and pd.notna(base_macro)
            else np.nan
        )

        gain_rows.append({
            "domain": domain,
            "comparison":
                f"b4_minus_{baseline}",
            "micro_f1_gain":
                float(
                    b4_micro
                    - base_micro
                ),
            "macro_f1_gain":
                macro_gain,
        })

transfer_gain_summary = pd.DataFrame(
    gain_rows
)

display(
    transfer_gain_summary
)


In [ ]:

all_predictions = pd.concat(
    [
        baseline_pred[
            [
                "model",
                "domain",
                "review_id",
                "true_aspects",
                "predicted_aspects",
            ]
        ],
        transfer_pred[
            [
                "model",
                "domain",
                "review_id",
                "true_aspects",
                "predicted_aspects",
            ]
        ],
    ],
    ignore_index=True,
)

all_predictions[
    "true_set"
] = (
    all_predictions[
        "true_aspects"
    ]
    .apply(
        lambda x:
            set(
                json.loads(x)
            )
    )
)

all_predictions[
    "pred_set"
] = (
    all_predictions[
        "predicted_aspects"
    ]
    .apply(
        lambda x:
            set(
                json.loads(x)
            )
    )
)


def example_f1(
    true_set,
    pred_set,
):
    if (
        len(true_set) == 0
        and len(pred_set) == 0
    ):
        return 1.0

    denominator = (
        len(true_set)
        + len(pred_set)
    )

    if denominator == 0:
        return 0.0

    return (
        2.0
        * len(
            true_set
            & pred_set
        )
        / denominator
    )


all_predictions[
    "example_f1"
] = [
    example_f1(
        true_set,
        pred_set,
    )
    for (
        true_set,
        pred_set,
    ) in zip(
        all_predictions[
            "true_set"
        ],
        all_predictions[
            "pred_set"
        ],
    )
]


def paired_bootstrap(
    domain,
    model_a,
    model_b,
    n_boot=2000,
    seed=42,
):
    a = (
        all_predictions[
            (
                all_predictions[
                    "domain"
                ]
                == domain
            )
            & (
                all_predictions[
                    "model"
                ]
                == model_a
            )
        ][
            [
                "review_id",
                "example_f1",
            ]
        ]
        .rename(
            columns={
                "example_f1":
                    "score_a"
            }
        )
    )

    b = (
        all_predictions[
            (
                all_predictions[
                    "domain"
                ]
                == domain
            )
            & (
                all_predictions[
                    "model"
                ]
                == model_b
            )
        ][
            [
                "review_id",
                "example_f1",
            ]
        ]
        .rename(
            columns={
                "example_f1":
                    "score_b"
            }
        )
    )

    paired = a.merge(
        b,
        on="review_id",
        how="inner",
    )

    if paired.empty:
        return None

    diff = (
        paired[
            "score_a"
        ]
        - paired[
            "score_b"
        ]
    ).to_numpy()

    rng = np.random.default_rng(
        seed
    )

    boot_means = np.empty(
        n_boot,
        dtype=float,
    )

    for i in range(n_boot):
        sample_idx = rng.integers(
            0,
            len(diff),
            size=len(diff),
        )

        boot_means[i] = (
            diff[
                sample_idx
            ]
            .mean()
        )

    ci_low, ci_high = (
        np.percentile(
            boot_means,
            [2.5, 97.5],
        )
    )

    p_value = np.nan

    if (
        SCIPY_AVAILABLE
        and np.any(
            diff != 0
        )
    ):
        try:
            p_value = float(
                wilcoxon(
                    diff
                ).pvalue
            )
        except Exception:
            p_value = np.nan

    return {
        "domain": domain,
        "comparison":
            f"{model_a}_vs_{model_b}",
        "n_paired_reviews":
            len(paired),
        "mean_example_f1_difference":
            float(
                diff.mean()
            ),
        "bootstrap_ci_2_5":
            float(ci_low),
        "bootstrap_ci_97_5":
            float(ci_high),
        "wilcoxon_p_value":
            p_value,
    }


stats_rows = []

for domain in [
    "phone",
    "laptop",
]:
    for baseline in [
        "b1",
        "b2",
        "b3",
    ]:
        result = paired_bootstrap(
            domain,
            "b4",
            baseline,
        )

        if result is not None:
            stats_rows.append(
                result
            )

paired_stats = pd.DataFrame(
    stats_rows
)

display(
    paired_stats
)


In [ ]:

b4_predictions = (
    all_predictions[
        all_predictions[
            "model"
        ].eq("b4")
    ]
    .copy()
)

error_counts = {}

for _, row in (
    b4_predictions
    .iterrows()
):
    true_set = row[
        "true_set"
    ]

    pred_set = row[
        "pred_set"
    ]

    for aspect in (
        true_set
        - pred_set
    ):
        key = (
            row["domain"],
            aspect,
            "false_negative",
        )

        error_counts[key] = (
            error_counts.get(
                key,
                0,
            )
            + 1
        )

    for aspect in (
        pred_set
        - true_set
    ):
        key = (
            row["domain"],
            aspect,
            "false_positive",
        )

        error_counts[key] = (
            error_counts.get(
                key,
                0,
            )
            + 1
        )

error_analysis = pd.DataFrame(
    [
        {
            "domain": domain,
            "canonical_aspect":
                aspect,
            "error_type":
                error_type,
            "count":
                count,
        }
        for (
            domain,
            aspect,
            error_type,
        ), count
        in error_counts.items()
    ]
)

if not error_analysis.empty:
    error_analysis = (
        error_analysis
        .sort_values(
            [
                "domain",
                "count",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )

display(
    error_analysis.head(
        30
    )
)


In [ ]:

sentiment_calibration_summary = (
    sentiment_metrics[
        sentiment_metrics[
            "segment"
        ].isin(
            [
                "overall",
                "phone",
                "laptop",
            ]
        )
    ]
    .copy()
)

display(
    sentiment_calibration_summary
)

display(
    group_metrics
)


In [ ]:

plot_df = (
    final_comparison[
        [
            "model",
            "segment",
            "micro_f1",
        ]
    ]
    .pivot(
        index="model",
        columns="segment",
        values="micro_f1",
    )
)

ax = plot_df.plot(
    kind="bar",
    figsize=(9, 5),
)

ax.set_title(
    "Aspect Detection Micro-F1 by Model and Domain"
)

ax.set_xlabel(
    "Model"
)

ax.set_ylabel(
    "Micro-F1"
)

ax.set_ylim(
    0,
    1,
)

plt.tight_layout()

plt.savefig(
    FINAL_DIR
    / "aspect_model_micro_f1.png",
    dpi=160,
)

plt.show()


cal_plot = (
    sentiment_calibration_summary[
        sentiment_calibration_summary[
            "segment"
        ].eq("overall")
    ][
        [
            "calibration",
            "ece",
            "brier",
            "nll",
        ]
    ]
    .set_index(
        "calibration"
    )
)

ax = cal_plot.plot(
    kind="bar",
    figsize=(8, 5),
)

ax.set_title(
    "Sentiment Confidence Calibration"
)

ax.set_xlabel(
    "Calibration"
)

ax.set_ylabel(
    "Metric value"
)

plt.tight_layout()

plt.savefig(
    FINAL_DIR
    / "sentiment_calibration_comparison.png",
    dpi=160,
)

plt.show()


group_plot = (
    group_metrics[
        [
            "model",
            "semantic_group",
            "micro_f1",
        ]
    ]
    .pivot(
        index="semantic_group",
        columns="model",
        values="micro_f1",
    )
)

ax = group_plot.plot(
    kind="bar",
    figsize=(9, 5),
)

ax.set_title(
    "B3 vs B4 by Semantic Group"
)

ax.set_xlabel(
    "Semantic group"
)

ax.set_ylabel(
    "Micro-F1"
)

ax.set_ylim(
    0,
    1,
)

plt.tight_layout()

plt.savefig(
    FINAL_DIR
    / "b3_b4_group_micro_f1.png",
    dpi=160,
)

plt.show()


In [ ]:

ablation_path = (
    TRANSFER_DIR
    / "b4_ablation_metrics.csv"
)

if ablation_path.exists():
    ablation_df = pd.read_csv(
        ablation_path
    )

    display(
        ablation_df
    )

    ablation_df.to_csv(
        FINAL_DIR
        / "b4_ablation_summary.csv",
        index=False,
    )
else:
    print(
        "Optional B4 no-gate ablation was not run. "
        "No ablation result is fabricated."
    )

final_comparison.to_csv(
    FINAL_DIR
    / "final_aspect_model_comparison.csv",
    index=False,
)

transfer_gain_summary.to_csv(
    FINAL_DIR
    / "transfer_gain_summary.csv",
    index=False,
)

paired_stats.to_csv(
    FINAL_DIR
    / "paired_bootstrap_significance.csv",
    index=False,
)

error_analysis.to_csv(
    FINAL_DIR
    / "b4_error_analysis.csv",
    index=False,
)

sentiment_calibration_summary.to_csv(
    FINAL_DIR
    / "sentiment_calibration_summary.csv",
    index=False,
)

required_outputs = [
    FINAL_DIR
    / "final_aspect_model_comparison.csv",

    FINAL_DIR
    / "transfer_gain_summary.csv",

    FINAL_DIR
    / "paired_bootstrap_significance.csv",

    FINAL_DIR
    / "b4_error_analysis.csv",

    FINAL_DIR
    / "sentiment_calibration_summary.csv",

    FINAL_DIR
    / "aspect_model_micro_f1.png",

    FINAL_DIR
    / "sentiment_calibration_comparison.png",

    FINAL_DIR
    / "b3_b4_group_micro_f1.png",
]

checks = {
    "all_four_aspect_models_present":
        {
            "b1",
            "b2",
            "b3",
            "b4",
        }.issubset(
            set(
                final_comparison[
                    "model"
                ]
            )
        ),

    "phone_and_laptop_present":
        {
            "phone",
            "laptop",
        }.issubset(
            set(
                final_comparison[
                    "segment"
                ]
            )
        ),

    "b4_group_results_present":
        "b4" in set(
            group_metrics[
                "model"
            ]
        ),

    "sentiment_calibration_present":
        {
            "uncalibrated",
            "calibrated",
        }.issubset(
            set(
                sentiment_calibration_summary[
                    "calibration"
                ]
            )
        ),

    "outputs_saved":
        all(
            path.exists()
            for path in required_outputs
        ),
}

validation_df = pd.DataFrame(
    [
        {
            "check": key,
            "passed": bool(value),
        }
        for (
            key,
            value,
        ) in checks.items()
    ]
)

display(
    validation_df
)

if not validation_df[
    "passed"
].all():
    failed = validation_df.loc[
        ~validation_df["passed"],
        "check",
    ].tolist()

    raise ValueError(
        f"FINAL ANALYSIS VALIDATION FAILED: {failed}"
    )

print(
    "C1 FINAL EVALUATION + ANALYSIS: PASSED"
)
